In [23]:
import os
import json
from dataclasses import dataclass
from typing import List, Dict
from pathlib import Path

In [24]:
@dataclass
class Keypoint:
    def __init__(self, id: str, x: float, y: float):
        self.id = id
        self.pixelPosition = (x, y)

@dataclass
class KeypointOnImage:
    def __init__(self, file: str, keypoints: List[Keypoint]):
        self.file = file
        self.keypoints = keypoints


@dataclass
class ModelOutput:
    def __init__(self, model: str, keypointOnImages: List[KeypointOnImage]):
        self.model = model
        self.keypointOnImages = keypointOnImages

@dataclass
class ModelComparison:
    def __init__(self, model: str, keypoint_errors: Dict[str, float]):
        self.model = model
        self.keypoint_errors = keypoint_errors


KEYPOINT_IDS = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

### Compare outputs of models based on groud truth from RTMO-X

In [33]:
TITLE = "Model Outputs Comparison of Biceps Curl Exercise"
OUTPUTS_DIR = Path(r"D:\Magistrska\blindoff-magistrska\fitcode-frontend-next\public\exercise-cut-videos-to-images\db-biceps-curl_frames_10fps\results")
GROUND_TRUTH_FILE_NAME = "RTMO_results.json"

# Find all .json files except ground truth
json_files = [
    f for f in os.listdir(OUTPUTS_DIR)
    if f.endswith(".json") and f != GROUND_TRUTH_FILE_NAME
]

# Read ground truth
with open(OUTPUTS_DIR / GROUND_TRUTH_FILE_NAME, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

# Read all model output files
model_outputs: List[dict] = []
for json_file in json_files:
    with open(OUTPUTS_DIR / json_file, "r", encoding="utf-8") as f:
        model_outputs.append(json.load(f))

# Build KEYPOINT_IDS automatically from ground truth (safer than hardcoding)
KEYPOINT_IDS = sorted({
    kp["id"]
    for item in ground_truth["keypointsOnImages"]
    for kp in item["keypoints"]
})

# Init model comparisons
model_comparisons = [
    {
        "model": out["model"],
        "keypoint_errors": {kp_id: 0.0 for kp_id in KEYPOINT_IDS},
    }
    for out in model_outputs
]

print(f"Comparing {len(model_outputs)} model outputs against ground truth from {ground_truth['model']}")

# Pre-index model outputs by file for fast lookup
# model_index[model_name][file] -> list of keypoints
model_index: Dict[str, Dict[str, List[dict]]] = {}
for out in model_outputs:
    per_file = {}
    for item in out.get("keypointsOnImages", []):
        per_file[item["file"]] = item["keypoints"]
    model_index[out["model"]] = per_file

# Iterate ground truth frames
for gt_item in ground_truth.get("keypointsOnImages", []):
    file = gt_item["file"]
    gt_keypoints = gt_item["keypoints"]

    # Compare each model vs this frame
    for comp in model_comparisons:
        model_name = comp["model"]
        model_keypoints = model_index.get(model_name, {}).get(file)

        if model_keypoints is None:
            print(f"No keypoints found for file {file} in model {model_name}")
            continue

        # Index model keypoints by id for O(1) access
        model_kp_by_id = {kp["id"]: kp for kp in model_keypoints}

        for gt_kp in gt_keypoints:
            kp_id = gt_kp["id"]
            model_kp = model_kp_by_id.get(kp_id)

            if model_kp is None:
                print(f"No keypoint {kp_id} found for file {file} in model {model_name}")
                continue

            dx = gt_kp["pixelPosition"]["x"] - model_kp["pixelPosition"]["x"]
            dy = gt_kp["pixelPosition"]["y"] - model_kp["pixelPosition"]["y"]
            error = (dx * dx + dy * dy) ** 0.5

            comp["keypoint_errors"][kp_id] += error

print(f"Model comparison results for {TITLE}:")
for comp in model_comparisons:
    print(f"Model: {comp['model']}")
    errors_sum = sum(comp["keypoint_errors"].values())
    print(f"  Total Keypoint Error: {errors_sum:.2f}")
    for keypoint_id, error in comp["keypoint_errors"].items():
        print(f"  Keypoint: {keypoint_id}, Total Error: {error:.2f}")
    print()

Comparing 12 model outputs against ground truth from RTMO_X
Model comparison results for Model Outputs Comparison of Biceps Curl Exercise:
Model: BlazePose Full
  Total Keypoint Error: 21.57
  Keypoint: left_ankle, Total Error: 1.77
  Keypoint: left_ear, Total Error: 0.84
  Keypoint: left_elbow, Total Error: 2.01
  Keypoint: left_eye, Total Error: 0.63
  Keypoint: left_hip, Total Error: 1.32
  Keypoint: left_knee, Total Error: 1.23
  Keypoint: left_shoulder, Total Error: 0.48
  Keypoint: left_wrist, Total Error: 1.76
  Keypoint: nose, Total Error: 0.83
  Keypoint: right_ankle, Total Error: 1.62
  Keypoint: right_ear, Total Error: 0.87
  Keypoint: right_elbow, Total Error: 2.37
  Keypoint: right_eye, Total Error: 0.51
  Keypoint: right_hip, Total Error: 1.26
  Keypoint: right_knee, Total Error: 1.47
  Keypoint: right_shoulder, Total Error: 1.01
  Keypoint: right_wrist, Total Error: 1.58

Model: BlazePose Heavy
  Total Keypoint Error: 19.29
  Keypoint: left_ankle, Total Error: 0.74
  Key